Write your solution in here.

- Your code has to reference all data files using **relative** paths so that they can be loaded on other computers without modifications.
- All data files need to be stored in the `data/` folder.

For example:

In [122]:
import pandas as pd

# Define the path to the data directory
DATA_PATH = './data'

# Read the CSV using a relative path
df = pd.read_csv(f'{DATA_PATH}/sce_extract_2015.csv')

# Term Paper 2 - Expectations about Macroeconomic and Financial Variables

## Part 1 - Data Preprocessing (SCE)

In [123]:
import pandas as pd
import numpy as np
import glob

DATA_PATH = './data'

# Variables asked only at the first interview → need forward-filling
COLS_FIRST_WAVE = [
    'owner', 'health', 'age_init',
    'num_lit_q1_correct', 'num_lit_q2_correct', 'num_lit_q3_correct',
    'num_lit_q5_correct', 'num_lit_q6_correct',
    'num_lit_q8_correct', 'num_lit_q9_correct',
]

COLS_EXPECTATIONS = ['infl_1y', 'house_price_change', 'prob_unrate_up', 'prob_stocks_up']

COLS_INDIVIDUAL = [
    'female', 'hispanic', 'black', 'educ',
    'hh_inc_bin_rank', 'working', 'couple', 'num_kids',
    'financial_past_12m', 'financial_12m', 'take_fin_risk'
]

COLS_META = ['userid', 'wid', 'date', 'tenure', 'weight']
ALL_COLS  = COLS_META + COLS_EXPECTATIONS + COLS_FIRST_WAVE + COLS_INDIVIDUAL

files = sorted(glob.glob(f'{DATA_PATH}/sce_extract_*.csv'))

df = pd.concat(
    [pd.read_csv(f, usecols=lambda c: c in ALL_COLS, parse_dates=['date']) for f in files],
    ignore_index=True
).sort_values(['userid', 'date']).reset_index(drop=True)

n_obs_init   = len(df)
n_ind_init   = df['userid'].nunique()
n_waves_init = df['wid'].nunique()

first_date_init = df['date'].min().date()
last_date_init  = df['date'].max().date()

print("=== Initial sample ===")
print(f"  Observations : {n_obs_init:,}")
print(f"  Individuals  : {n_ind_init:,}")
print(f"  Survey waves : {n_waves_init}")
print(f"  Date range   : {first_date_init} → {last_date_init}")

=== Initial sample ===
  Observations : 180,268
  Individuals  : 23,886
  Survey waves : 143
  Date range   : 2013-06-01 → 2025-05-01


In [124]:
# Forward-fill within each individual (only fills NaNs after the first observed value)
df[COLS_FIRST_WAVE] = (
    df.groupby('userid')[COLS_FIRST_WAVE]
    .transform(lambda s: s.ffill())
)

In [ ]:
# Drop observations from 2025 due to Trump-volatility
mask_2025 = df['date'].dt.year == 2025
n_dropped_2025 = mask_2025.sum()
df = df[~mask_2025].copy()
print(f"Step 3 — Dropped {n_dropped_2025:,} observations from 2025")

Step 3 — Dropped 4,167 observations from 2025


In [ ]:
# Final set of columns to use in the analysis, before NA analysis
ANALYSIS_COLS = COLS_EXPECTATIONS + COLS_FIRST_WAVE + COLS_INDIVIDUAL

df[ANALYSIS_COLS].isna().agg(['sum', 'mean']).T.rename(columns={'sum': 'Count NA', 'mean': 'Share NA'}).sort_values('Share NA', ascending=False)

,Count NA,Share NA
num_lit_q9_correct,36718.0,0.208505
num_lit_q8_correct,36396.0,0.206677
take_fin_risk,36355.0,0.206444
health,36342.0,0.206370
couple,12992.0,0.073776
hh_inc_bin_rank,1631.0,0.009262
num_lit_q6_correct,1053.0,0.005980
prob_stocks_up,960.0,0.005451
educ,742.0,0.004213
infl_1y,682.0,0.003873


In [ ]:
# Decide which columns to keep based on the NA analysis and substantive importance
COLS_FIRST_WAVE_USED = [
    'owner', 'age_init',
    'num_lit_q1_correct', 'num_lit_q2_correct', 'num_lit_q3_correct',
    'num_lit_q5_correct', 'num_lit_q6_correct'
    # maybe keep 'health' if you think it’s substantively important?
]

COLS_INDIVIDUAL_USED = [
    'female', 'black', 'hispanic', 'educ',
    'hh_inc_bin_rank', 'working', 'couple',
    'num_kids',
    'financial_past_12m', 'financial_12m'
    # consider whether to keep 'take_fin_risk'
]

# Final set of columns to use in the analysis
ANALYSIS_COLS = COLS_EXPECTATIONS + COLS_FIRST_WAVE_USED + COLS_INDIVIDUAL_USED

In [130]:


n_before = len(df)
df = df.dropna(subset=ANALYSIS_COLS).reset_index(drop=True)
print(f"Step 4 — Dropped {n_before - len(df):,} observations with missing values")

Step 4 — Dropped 16,966 observations with missing values


In [131]:
for col in COLS_EXPECTATIONS:
    p1  = df[col].quantile(0.01)
    p99 = df[col].quantile(0.99)
    n_before = len(df)
    df = df[(df[col] > p1) & (df[col] < p99)]
    print(f"  {col}: P1={p1:.2f}, P99={p99:.2f} → dropped {n_before - len(df):,} obs")

df = df.reset_index(drop=True)

  infl_1y: P1=-30.00, P99=60.00 → dropped 3,400 obs
  house_price_change: P1=-20.00, P99=40.00 → dropped 4,406 obs
  prob_unrate_up: P1=0.00, P99=99.00 → dropped 4,281 obs
  prob_stocks_up: P1=1.00, P99=90.00 → dropped 4,781 obs


In [132]:
df['optimist_unrate']      = (df['prob_unrate_up'] < 50).astype(int)
df['optimist_stocks']      = (df['prob_stocks_up'] > 50).astype(int)
df['optimist_house_price'] = (df['house_price_change'] > 0).astype(int)

summary = pd.DataFrame({
    'Sample'       : ['Initial', 'Final'],
    'Observations' : [n_obs_init, len(df)],
    'Individuals'  : [n_ind_init, df['userid'].nunique()],
    'Survey waves' : [n_waves_init, df['wid'].nunique()],
    'First date'   : [first_date_init, df['date'].min().date()],
    'Last date'    : [last_date_init, df['date'].max().date()],
})
display(summary)

,Sample,Observations,Individuals,Survey waves,First date,Last date
0,Initial,180268,23886,143,2013-06-01,2025-05-01
1,Final,142267,21628,139,2013-06-01,2024-12-31


## Part 2 - Data Preprocessing (Macro/Finance)